# Imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import random

# Data loading, splitting & encoding
This block loads the data, creates the chronological split, and maps all strings to contiguous integers.

In [4]:
# --- 1. Load Data ---
interactions_path = Path.cwd().parent/"data"/"interactions_train.csv"
items_path = Path.cwd().parent/"data"/"items.csv"

interactions_df = pd.read_csv(interactions_path)
interactions_df = interactions_df.rename(columns={"u": "user_id", "i": "item_id", "t": "timestamp"})

items_df = pd.read_csv(items_path)
items_df = items_df.rename(columns={"i" : "item_id"})

# --- 2. Temporal 80/20 Split ---
interactions_df = interactions_df.sort_values(by=['user_id', 'timestamp']).reset_index(drop=True)
interactions_df['interaction_rank'] = interactions_df.groupby('user_id')['timestamp'].rank(method='first', ascending=True)
interactions_df['total_interactions'] = interactions_df.groupby('user_id')['timestamp'].transform('count')

train_df = interactions_df[interactions_df['interaction_rank'] <= interactions_df['total_interactions'] * 0.8].copy()
val_df = interactions_df[interactions_df['interaction_rank'] > interactions_df['total_interactions'] * 0.8].copy()

# --- 3. Encoding Metadata ---
items_df['Author'] = items_df['Author'].fillna('Unknown')
items_df['Publisher'] = items_df['Publisher'].fillna('Unknown')
items_df['Subjects'] = items_df['Subjects'].fillna('Unknown')

# Create mappings based on ALL data to prevent "Unknown ID" errors during inference
user2idx = {u: i for i, u in enumerate(interactions_df['user_id'].unique())}
idx2user = {i: u for i, u in enumerate(interactions_df['user_id'].unique())}
item2idx = {i: idx for idx, i in enumerate(items_df['item_id'].unique())}
idx2item = {idx: i for idx, i in enumerate(items_df['item_id'].unique())}

author2idx = {a: i for i, a in enumerate(items_df['Author'].unique())}
publisher2idx = {p: i for i, p in enumerate(items_df['Publisher'].unique())}
topic2idx = {t: i for i, t in enumerate(items_df['Subjects'].unique())}

num_users, num_items = len(user2idx), len(item2idx)

# Map DataFrames
train_df['user_idx'] = train_df['user_id'].map(user2idx)
train_df['item_idx'] = train_df['item_id'].map(item2idx)
val_df['user_idx'] = val_df['user_id'].map(user2idx)
val_df['item_idx'] = val_df['item_id'].map(item2idx)

# --- 4. Fast Lookup Tensors & History Dictionaries ---
item_metadata_lookup = np.zeros((num_items, 3), dtype=np.int64)
for _, row in items_df.iterrows():
    if row['item_id'] in item2idx:
        i_idx = item2idx[row['item_id']]
        item_metadata_lookup[i_idx, 0] = author2idx[row['Author']]
        item_metadata_lookup[i_idx, 1] = publisher2idx[row['Publisher']]
        item_metadata_lookup[i_idx, 2] = topic2idx[row['Subjects']]

item_metadata_tensor = torch.tensor(item_metadata_lookup, dtype=torch.long)

train_user_history = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()
val_user_history = val_df.groupby('user_idx')['item_idx'].apply(list).to_dict()
full_user_history = interactions_df.groupby('user_id')['item_id'].apply(set).to_dict()

# Dataset & Model Architecture
We use the explicit while-loop for exact negative sampling and include heavy regularizations (BatchNorm1d and Dropout) to combat the sparsity.

In [5]:
class ExplicitBPRDataset(Dataset):
    def __init__(self, df, user_history, num_items):
        self.users = df['user_idx'].values
        self.pos_items = df['item_idx'].values
        self.user_history = user_history
        self.num_items = num_items

    def __len__(self): return len(self.users)

    def __getitem__(self, idx):
        user = self.users[idx]
        pos_item = self.pos_items[idx]
        
        neg_item = random.randint(0, self.num_items - 1)
        while neg_item in self.user_history.get(user, set()):
            neg_item = random.randint(0, self.num_items - 1)
            
        return (
            torch.tensor(user, dtype=torch.long), 
            torch.tensor(pos_item, dtype=torch.long), 
            torch.tensor(neg_item, dtype=torch.long)
        )

class TwoTowerRecommender(nn.Module):
    def __init__(self, num_users, num_items, num_authors, num_publishers, num_topics, item_metadata, embed_dim=32):
        super(TwoTowerRecommender, self).__init__()
        self.register_buffer('item_metadata', item_metadata)
        
        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.item_emb = nn.Embedding(num_items, embed_dim)
        self.author_emb = nn.Embedding(num_authors, embed_dim // 2)
        self.publisher_emb = nn.Embedding(num_publishers, embed_dim // 2)
        self.topic_emb = nn.Embedding(num_topics, embed_dim // 2)
        
        self.item_bias = nn.Embedding(num_items, 1)
        nn.init.zeros_(self.item_bias.weight)
        
        concat_dim = embed_dim + (embed_dim // 2) * 3
        
        # Heavy regularization for sparse data
        self.item_mlp = nn.Sequential(
            nn.Linear(concat_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(128, embed_dim)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding) and m != self.item_bias:
                nn.init.normal_(m.weight, std=0.01)

    def get_item_representation(self, item_indices):
        i_emb = self.item_emb(item_indices)
        meta = self.item_metadata[item_indices]
        a_emb, p_emb, t_emb = self.author_emb(meta[:, 0]), self.publisher_emb(meta[:, 1]), self.topic_emb(meta[:, 2])
        concat_emb = torch.cat([i_emb, a_emb, p_emb, t_emb], dim=-1)
        return self.item_mlp(concat_emb)

    def forward(self, user, pos_item, neg_item):
        u_emb = self.user_emb(user)
        pos_i_emb = self.get_item_representation(pos_item)
        neg_i_emb = self.get_item_representation(neg_item)
        
        pos_score = (u_emb * pos_i_emb).sum(dim=1) + self.item_bias(pos_item).squeeze()
        neg_score = (u_emb * neg_i_emb).sum(dim=1) + self.item_bias(neg_item).squeeze()
        return pos_score, neg_score

# Evaluation & Training Loop
This trains the model, tracks validation P@10 directly on your 20% holdout set, and saves the best model states.

In [6]:
def evaluate_precision_at_k(model, val_user_history, train_user_history, device, k=10):
    model.eval()
    precisions = []
    with torch.no_grad():
        all_items = torch.arange(num_items, device=device)
        all_item_embs = model.get_item_representation(all_items)
        all_item_biases = model.item_bias(all_items).squeeze()
        
        for user_idx, true_items in val_user_history.items():
            u_tensor = torch.tensor([user_idx], device=device)
            u_emb = model.user_emb(u_tensor)
            
            scores = torch.matmul(u_emb, all_item_embs.T).squeeze() + all_item_biases
            
            past_items = list(train_user_history.get(user_idx, []))
            if past_items:
                scores[past_items] = -float('inf')
                
            top_k_items = torch.topk(scores, k=k).indices.cpu().numpy()
            hits = len(set(top_k_items).intersection(set(true_items)))
            precisions.append(hits / k)
            
    return np.mean(precisions)

# --- Training Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on {device}")

# Explicit CPU loader - num_workers=0 prevents freezing
train_dataset = ExplicitBPRDataset(train_df, train_user_history, num_items)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, num_workers=0)

# Sparsity-optimized hyperparameters
embed_dim = 32
learning_rate = 0.001
weight_decay = 1e-3
epochs = 15

model = TwoTowerRecommender(
    num_users=num_users, num_items=num_items, 
    num_authors=len(author2idx), num_publishers=len(publisher2idx), num_topics=len(topic2idx),
    item_metadata=item_metadata_tensor, embed_dim=embed_dim
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

best_val_prec = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for user, pos_item, neg_item in train_loader:
        user, pos_item, neg_item = user.to(device), pos_item.to(device), neg_item.to(device)
        
        optimizer.zero_grad()
        pos_score, neg_score = model(user, pos_item, neg_item)
        loss = -torch.nn.functional.logsigmoid(pos_score - neg_score).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    val_prec = evaluate_precision_at_k(model, val_user_history, train_user_history, device, k=10)
    
    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_loss:.4f} | Val P@10: {val_prec:.5f}")
    
    if val_prec > best_val_prec:
        best_val_prec = val_prec
        torch.save(model.state_dict(), 'best_twotower_method2.pth')

print(f"\nTraining complete. Best Validation P@10: {best_val_prec:.5f}")

Training on cuda
Epoch 01/15 | Train Loss: 0.6690 | Val P@10: 0.00047
Epoch 02/15 | Train Loss: 0.4995 | Val P@10: 0.00073
Epoch 03/15 | Train Loss: 0.3571 | Val P@10: 0.00055
Epoch 04/15 | Train Loss: 0.2412 | Val P@10: 0.00078
Epoch 05/15 | Train Loss: 0.1630 | Val P@10: 0.00092
Epoch 06/15 | Train Loss: 0.1238 | Val P@10: 0.00087
Epoch 07/15 | Train Loss: 0.1055 | Val P@10: 0.00082
Epoch 08/15 | Train Loss: 0.1024 | Val P@10: 0.00085
Epoch 09/15 | Train Loss: 0.0874 | Val P@10: 0.00105
Epoch 10/15 | Train Loss: 0.0776 | Val P@10: 0.00102
Epoch 11/15 | Train Loss: 0.0850 | Val P@10: 0.00100
Epoch 12/15 | Train Loss: 0.0810 | Val P@10: 0.00140
Epoch 13/15 | Train Loss: 0.0713 | Val P@10: 0.00134
Epoch 14/15 | Train Loss: 0.0748 | Val P@10: 0.00128
Epoch 15/15 | Train Loss: 0.0767 | Val P@10: 0.00137

Training complete. Best Validation P@10: 0.00140


# Generate the Final Submission
This final block loads the best saved weights, predicts the top 10 books per user (masking out all known past history), and strictly formats the CSV to pass the Kaggle grading bot constraints.

In [7]:
# 1. Load the best saved model
model.load_state_dict(torch.load('best_twotower_method2.pth', map_location=device))
model.eval()

recommendations = []

with torch.no_grad():
    print("Pre-computing representations for submission")
    all_items = torch.arange(num_items, device=device)
    all_item_embs = model.get_item_representation(all_items)
    all_item_biases = model.item_bias(all_items).squeeze()
    
    print("Generating predictions")
    for user_idx in range(num_users):
        u_tensor = torch.tensor([user_idx], device=device)
        u_emb = model.user_emb(u_tensor)
        
        scores = torch.matmul(u_emb, all_item_embs.T).squeeze() + all_item_biases
        
        # Mask out ALL history (train + val) so we only recommend genuinely new books
        read_items = list(full_user_history.get(user_idx, set()))
        if read_items:
            scores[read_items] = -float('inf')
            
        top_10_idx = torch.topk(scores, k=10).indices.cpu().numpy()
        
        original_user_id = idx2user[user_idx]
        top_10_item_ids = [idx2item[idx] for idx in top_10_idx]
        
        recommendations.append({
            'user_id': original_user_id,
            'top_10': " ".join([str(i) for i in top_10_item_ids]) 
        })

# 2. Format rigidly for Kaggle
submission_df = pd.DataFrame(recommendations)
submission_df['user_id'] = pd.to_numeric(submission_df['user_id'])
submission_df = submission_df.sort_values(by='user_id').reset_index(drop=True)

submission_df.to_csv(Path.cwd().parent/"submissions"/'submission_twotowers.csv', index=False)

Pre-computing representations for submission
Generating predictions
